# BERT for Extractive Question Answering (SQuAD v1.1)

## 1. Setup

In [1]:
import collections
import re
import string
import time

import random
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from torch.utils.data import DataLoader
from transformers import (
    AutoModel,
    AutoTokenizer,
    BertForQuestionAnswering,
    Trainer,
    TrainerCallback,
    TrainingArguments,
    default_data_collator,
)
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

/home/giobbva/BERT/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
SEED = 42


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(SEED)

MODEL_NAME = "bert-base-uncased"
N_TRAIN_EXAMPLES = 15_000
MAX_LENGTH = 384        # question + context window
STRIDE = 128            # overlap between windows of long contexts
IGNORE_INDEX = -100

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__} | device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch 2.6.0+cu124 | device: cuda
GPU: NVIDIA GeForce RTX 4060 Laptop GPU


## 2. Data Loading

In [3]:
raw_datasets = load_dataset("rajpurkar/squad")

train_examples = raw_datasets["train"].shuffle(seed=SEED).select(range(N_TRAIN_EXAMPLES))
eval_examples = raw_datasets["validation"]

print(f"train examples     : {len(train_examples):,} (subsampled from {len(raw_datasets['train']):,})")
print(f"validation examples: {len(eval_examples):,}")
train_examples[0]

train examples     : 15,000 (subsampled from 87,599)
validation examples: 10,570


{'id': '573173d8497a881900248f0c',
 'title': 'Egypt',
 'context': 'The Pew Forum on Religion & Public Life ranks Egypt as the fifth worst country in the world for religious freedom. The United States Commission on International Religious Freedom, a bipartisan independent agency of the US government, has placed Egypt on its watch list of countries that require close monitoring due to the nature and extent of violations of religious freedom engaged in or tolerated by the government. According to a 2010 Pew Global Attitudes survey, 84% of Egyptians polled supported the death penalty for those who leave Islam; 77% supported whippings and cutting off of hands for theft and robbery; and 82% support stoning a person who commits adultery.',
 'question': 'What percentage of Egyptians polled support death penalty for those leaving Islam?',
 'answers': {'text': ['84%'], 'answer_start': [468]}}

In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("Fast tokenizer:", tokenizer.is_fast)

Fast tokenizer: True


## 3. Tokenization & Alignment

In [5]:
def prepare_features(examples):
    encoded = tokenizer(
        [q.strip() for q in examples["question"]],
        examples["context"],
        truncation="only_second",
        max_length=MAX_LENGTH,
        stride=STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )
    sample_map = encoded.pop("overflow_to_sample_mapping")
    offset_mapping = encoded.pop("offset_mapping")

    columns = ["example_id", "start_labels", "end_labels", "start_positions", "end_positions",
               "has_answer", "word_start_char", "word_end_char"]
    out = {name: [] for name in columns}

    for i, offsets in enumerate(offset_mapping):
        example_idx = sample_map[i]
        sequence_ids = encoded.sequence_ids(i)   # None = special, 0 = question, 1 = context
        word_ids = encoded.word_ids(i)

        answer = examples["answers"][example_idx]
        answer_start = answer["answer_start"][0]
        answer_end = answer_start + len(answer["text"][0])

        # character where each context word ends (= end of its last sub-token)
        word_end = {}
        for tok, (word_id, seq_id) in enumerate(zip(word_ids, sequence_ids)):
            if seq_id == 1:
                word_end[word_id] = offsets[tok][1]

        start_labels, end_labels, word_start_char, word_end_char = [], [], [], []
        start_position = end_position = 0
        previous_word_id = None
        for tok, (word_id, seq_id) in enumerate(zip(word_ids, sequence_ids)):
            is_first_subword = seq_id == 1 and word_id != previous_word_id
            previous_word_id = word_id if seq_id == 1 else None

            if not is_first_subword:
                # [CLS], [SEP], padding, question tokens, continuation sub-tokens
                start_labels.append(IGNORE_INDEX)
                end_labels.append(IGNORE_INDEX)
                word_start_char.append(-1)
                word_end_char.append(-1)
                continue

            w_start, w_end = offsets[tok][0], word_end[word_id]
            is_start = w_start <= answer_start < w_end
            is_end = w_start < answer_end <= w_end
            start_labels.append(int(is_start))
            end_labels.append(int(is_end))
            word_start_char.append(w_start)
            word_end_char.append(w_end)
            if is_start:
                start_position = tok
            if is_end:
                end_position = tok

        has_answer = 1 in start_labels and 1 in end_labels
        out["example_id"].append(examples["id"][example_idx])
        out["start_labels"].append(start_labels)
        out["end_labels"].append(end_labels)
        out["start_positions"].append(start_position if has_answer else 0)
        out["end_positions"].append(end_position if has_answer else 0)
        out["has_answer"].append(has_answer)
        out["word_start_char"].append(word_start_char)
        out["word_end_char"].append(word_end_char)

    encoded.update(out)
    return encoded


train_features = train_examples.map(
    prepare_features, batched=True, remove_columns=train_examples.column_names)
eval_features = eval_examples.map(
    prepare_features, batched=True, remove_columns=eval_examples.column_names)

n_train_windows = len(train_features)
train_features = train_features.filter(lambda f: f["has_answer"])   # windows that contain the answer
eval_loss_features = eval_features.filter(lambda f: f["has_answer"])  # for the validation loss

print(f"train windows: {n_train_windows:,} -> {len(train_features):,} containing the answer")
print(f"eval windows : {len(eval_features):,} ({len(eval_loss_features):,} containing the answer)")

train windows: 15,141 -> 15,016 containing the answer
eval windows : 10,753 (10,578 containing the answer)


In [6]:
# Sanity check: one batch, token by token
MODEL_INPUTS = ["input_ids", "token_type_ids", "attention_mask"]
LABEL_COLUMNS = ["start_positions", "end_positions", "start_labels", "end_labels"]

loader = DataLoader(train_features.select_columns(MODEL_INPUTS + LABEL_COLUMNS),
                    batch_size=2, collate_fn=default_data_collator)
batch = next(iter(loader))
print({name: tuple(tensor.shape) for name, tensor in batch.items()})

for row in range(len(batch["input_ids"])):
    tokens = tokenizer.convert_ids_to_tokens(batch["input_ids"][row])
    start_labels = batch["start_labels"][row].tolist()
    end_labels = batch["end_labels"][row].tolist()
    token_types = batch["token_type_ids"][row].tolist()
    print(f"\n=== feature {row} | start_position={batch['start_positions'][row].item()} "
          f"end_position={batch['end_positions'][row].item()} ===")
    print(f"{'pos':>3} | {'token':<15} | {'start':>5} | {'end':>5} | role")
    n_pad = 0
    for pos, (token, s, e, tt) in enumerate(zip(tokens, start_labels, end_labels, token_types)):
        if token == tokenizer.pad_token:
            n_pad += 1
            continue
        if token in tokenizer.all_special_tokens:
            role = "special"
        elif tt == 0:
            role = "question"
        elif s == IGNORE_INDEX:
            role = "continuation sub-token"
        else:
            role = "first sub-token of a context word"
        if s == 1 or e == 1:
            role += "  <-- ANSWER " + ("START" if s == 1 else "") + (" END" if e == 1 else "")
        print(f"{pos:>3} | {token:<15} | {s:>5} | {e:>5} | {role}")
    print(f"    + {n_pad} [PAD] positions (start/end = -100)")

pad_mask = batch["attention_mask"] == 0
assert (batch["start_labels"][pad_mask] == IGNORE_INDEX).all()
assert (batch["end_labels"][pad_mask] == IGNORE_INDEX).all()
assert (batch["start_labels"][:, 0] == IGNORE_INDEX).all()   # [CLS]
print("\nOK: [CLS], [SEP], question tokens, continuation sub-tokens and padding are all -100.")

{'input_ids': (2, 384), 'token_type_ids': (2, 384), 'attention_mask': (2, 384), 'start_positions': (2,), 'end_positions': (2,), 'start_labels': (2, 384), 'end_labels': (2, 384)}

=== feature 0 | start_position=97 end_position=98 ===
pos | token           | start |   end | role
  0 | [CLS]           |  -100 |  -100 | special
  1 | what            |  -100 |  -100 | question
  2 | percentage      |  -100 |  -100 | question
  3 | of              |  -100 |  -100 | question
  4 | egyptians       |  -100 |  -100 | question
  5 | polled          |  -100 |  -100 | question
  6 | support         |  -100 |  -100 | question
  7 | death           |  -100 |  -100 | question
  8 | penalty         |  -100 |  -100 | question
  9 | for             |  -100 |  -100 | question
 10 | those           |  -100 |  -100 | question
 11 | leaving         |  -100 |  -100 | question
 12 | islam           |  -100 |  -100 | question
 13 | ?               |  -100 |  -100 | question
 14 | [SEP]           |  -100 |  -100

In [7]:
# Corpus check: the span between the labelled start/end words must reproduce the gold answer
train_by_id = {ex["id"]: ex for ex in train_examples}
exact_matches = 0
for f in train_features.select_columns(
        ["example_id", "start_positions", "end_positions", "word_start_char", "word_end_char"]):
    ex = train_by_id[f["example_id"]]
    span = ex["context"][f["word_start_char"][f["start_positions"]]:f["word_end_char"][f["end_positions"]]]
    exact_matches += span == ex["answers"]["text"][0]

print(f"labelled span == gold answer text: {exact_matches:,} / {len(train_features):,} "
      f"({100 * exact_matches / len(train_features):.2f}%)")
print("(the rest are answers that start or end in the middle of a word, e.g. '1990' inside '1990s')")

labelled span == gold answer text: 14,976 / 15,016 (99.73%)
(the rest are answers that start or end in the middle of a word, e.g. '1990' inside '1990s')


## 4. Model 1: Feature-Based (frozen BERT + Logistic Regression)

In [8]:
bert_encoder = AutoModel.from_pretrained(MODEL_NAME)
for param in bert_encoder.parameters():
    param.requires_grad = False
bert_encoder.eval().to(device)

trainable = sum(p.numel() for p in bert_encoder.parameters() if p.requires_grad)
print(f"BERT parameters: {sum(p.numel() for p in bert_encoder.parameters()):,} | trainable: {trainable:,}")
assert trainable == 0

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 12024.47it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BERT parameters: 109,482,240 | trainable: 0


In [9]:
@torch.no_grad()
def encode(batch):
    inputs = {name: batch[name].to(device) for name in MODEL_INPUTS}
    with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == "cuda"):
        hidden = bert_encoder(**inputs).last_hidden_state
    return hidden.float().cpu()                                  # (batch, 384, 768)


NEGATIVES_PER_WINDOW = 10   # context words that are neither start nor end, sampled per window
generator = torch.Generator().manual_seed(SEED)

loader = DataLoader(train_features.select_columns(MODEL_INPUTS + LABEL_COLUMNS),
                    batch_size=64, collate_fn=default_data_collator)
X_parts, y_start_parts, y_end_parts = [], [], []

start = time.time()
for batch in loader:
    hidden = encode(batch)
    for i in range(len(hidden)):
        start_labels, end_labels = batch["start_labels"][i], batch["end_labels"][i]
        context_positions = (start_labels != IGNORE_INDEX).nonzero().squeeze(1)   # -100 excluded
        negatives = context_positions[(start_labels[context_positions] == 0) & (end_labels[context_positions] == 0)]
        negatives = negatives[torch.randperm(len(negatives), generator=generator)[:NEGATIVES_PER_WINDOW]]
        positions = torch.cat([batch["start_positions"][i:i + 1], batch["end_positions"][i:i + 1], negatives]).unique()
        X_parts.append(hidden[i, positions].numpy())
        y_start_parts.append(start_labels[positions].numpy())
        y_end_parts.append(end_labels[positions].numpy())

X_train = np.concatenate(X_parts)
y_start = np.concatenate(y_start_parts)
y_end = np.concatenate(y_end_parts)
m1_extract_seconds = time.time() - start
print(f"feature extraction: {m1_extract_seconds:.1f}s | X_train {X_train.shape} | "
      f"start positives {y_start.sum():,} | end positives {y_end.sum():,}")

feature extraction: 77.5s | X_train (175694, 768) | start positives 15,016 | end positives 15,016


In [10]:
start = time.time()
start_classifier = make_pipeline(StandardScaler(), LogisticRegression(max_iter=300, random_state=SEED))
start_classifier.fit(X_train, y_start)
end_classifier = make_pipeline(StandardScaler(), LogisticRegression(max_iter=300, random_state=SEED))
end_classifier.fit(X_train, y_end)
m1_fit_seconds = time.time() - start
print(f"start + end Logistic Regression fitted in {m1_fit_seconds:.1f}s")

start + end Logistic Regression fitted in 12.5s


In [11]:
# Start/end logits for every context word of the evaluation windows
eval_loader = DataLoader(eval_features.select_columns(MODEL_INPUTS + ["start_labels"]),
                         batch_size=64, collate_fn=default_data_collator)
m1_start_logits = np.full((len(eval_features), MAX_LENGTH), -1e9, dtype=np.float32)
m1_end_logits = np.full((len(eval_features), MAX_LENGTH), -1e9, dtype=np.float32)

start, row = time.time(), 0
for batch in eval_loader:
    hidden = encode(batch)
    valid = batch["start_labels"] != IGNORE_INDEX
    X = hidden[valid].numpy()
    rows, cols = (index.numpy() for index in valid.nonzero(as_tuple=True))
    m1_start_logits[rows + row, cols] = start_classifier.decision_function(X)
    m1_end_logits[rows + row, cols] = end_classifier.decision_function(X)
    row += len(hidden)
print(f"Model 1 inference on {len(eval_features):,} windows: {time.time() - start:.1f}s")

del bert_encoder, X_train, X_parts
torch.cuda.empty_cache()

Model 1 inference on 10,753 windows: 59.5s


## 5. Model 2: Full Fine-Tuning (BertForQuestionAnswering)

In [12]:
set_seed(SEED)
model = BertForQuestionAnswering.from_pretrained(MODEL_NAME)

ENCODER_LR = 2e-5
HEAD_LR = 1e-3

encoder_params = [p for name, p in model.named_parameters() if name.startswith("bert.")]
head_params = [p for name, p in model.named_parameters() if not name.startswith("bert.")]   # qa_outputs.*

optimizer = torch.optim.AdamW(
    [
        {"params": encoder_params, "lr": ENCODER_LR},
        {"params": head_params, "lr": HEAD_LR},
    ],
    weight_decay=0.01,
)
print(f"encoder: {sum(p.numel() for p in encoder_params):>12,} params @ lr={ENCODER_LR}")
print(f"QA head: {sum(p.numel() for p in head_params):>12,} params @ lr={HEAD_LR}")

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 4073.44it/s]
[transformers] BertForQuestionAnswering LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
qa_outputs.weight                          | MISSING    | 
qa_outputs.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not 

encoder:  108,891,648 params @ lr=2e-05
QA head:        1,538 params @ lr=0.001


In [13]:
class EpochTimer(TrainerCallback):
    def __init__(self):
        self.epoch_seconds = []

    def on_epoch_begin(self, args, state, control, **kwargs):
        self._start = time.time()

    def on_epoch_end(self, args, state, control, **kwargs):
        self.epoch_seconds.append(time.time() - self._start)


training_args = TrainingArguments(
    output_dir="bert-qa-checkpoints",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    fp16=True,
    lr_scheduler_type="linear",
    warmup_steps=0.1,                  # < 1 -> ratio of total steps
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    logging_steps=50,
    seed=SEED,
    report_to="none",
)

epoch_timer = EpochTimer()
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_features.select_columns(MODEL_INPUTS + ["start_positions", "end_positions"]),
    eval_dataset=eval_loss_features.select_columns(MODEL_INPUTS + ["start_positions", "end_positions"]),
    data_collator=default_data_collator,
    processing_class=tokenizer,
    optimizers=(optimizer, None),      # our two parameter groups; Trainer adds the scheduler
    callbacks=[epoch_timer],
)

In [14]:
train_result = trainer.train()

steps = trainer.state.global_step
runtime = train_result.metrics["train_runtime"]
print(f"\ntraining loss (average): {train_result.training_loss:.4f}")
print(f"total time: {runtime:.1f}s | steps: {steps} | seconds/step: {runtime / steps:.3f}")
for epoch, seconds in enumerate(epoch_timer.epoch_seconds, 1):
    print(f"epoch {epoch}: {seconds:.1f}s")

Epoch,Training Loss,Validation Loss
1,1.412012,1.354388
2,1.171735,1.296916


Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.24s/it]
[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'ber


training loss (average): 1.6834
total time: 613.4s | steps: 1878 | seconds/step: 0.327
epoch 1: 257.8s
epoch 2: 256.3s


In [15]:
log = pd.DataFrame(trainer.state.log_history)
train_log = log.dropna(subset=["loss"])[["step", "epoch", "loss", "learning_rate"]]
eval_log = log.dropna(subset=["eval_loss"])[["epoch", "eval_loss"]]
print(eval_log.to_string(index=False))
train_log.tail(10)

 epoch  eval_loss
   1.0   1.354388
   2.0   1.296916


,step,epoch,loss,learning_rate
28,1400,1.490948,1.049177,5.704142e-06
29,1450,1.544196,1.083611,5.112426e-06
30,1500,1.597444,1.086207,4.520710e-06
31,1550,1.650692,1.001299,3.928994e-06
32,1600,1.703940,1.099770,3.337278e-06
33,1650,1.757188,1.117108,2.745562e-06
34,1700,1.810437,1.038806,2.153846e-06
35,1750,1.863685,1.149922,1.562130e-06
36,1800,1.916933,1.118726,9.704142e-07
37,1850,1.970181,1.171735,3.786982e-07


In [16]:
FINAL_MODEL_DIR = "bert-qa-final"
trainer.save_model(FINAL_MODEL_DIR)            # best epoch (lowest validation loss)
tokenizer.save_pretrained(FINAL_MODEL_DIR)
print(f"best model saved to '{FINAL_MODEL_DIR}/'")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.88it/s]

best model saved to 'bert-qa-final/'


## 6. Evaluation (Exact Match & F1)

In [17]:
def extract_answers(examples, features, start_logits, end_logits, n_best=20, max_answer_tokens=30):
    word_start_char = np.array(features["word_start_char"])
    word_end_char = np.array(features["word_end_char"])
    windows_per_example = collections.defaultdict(list)
    for i, example_id in enumerate(features["example_id"]):
        windows_per_example[example_id].append(i)

    predictions = {}
    for example in examples:
        best_score, best_text = -np.inf, ""
        for i in windows_per_example[example["id"]]:
            valid = word_start_char[i] >= 0            # first sub-token of a context word
            start_scores = np.where(valid, start_logits[i], -np.inf)
            end_scores = np.where(valid, end_logits[i], -np.inf)
            for s in np.argsort(start_scores)[-n_best:]:
                for e in np.argsort(end_scores)[-n_best:]:
                    if not (valid[s] and valid[e]) or e < s or e - s + 1 > max_answer_tokens:
                        continue
                    score = start_scores[s] + end_scores[e]
                    if score > best_score:
                        best_score = score
                        best_text = example["context"][word_start_char[i][s]:word_end_char[i][e]]
        predictions[example["id"]] = best_text
    return predictions


# Official SQuAD v1.1 metric
def normalize_answer(text):
    text = text.lower()
    text = "".join(ch for ch in text if ch not in set(string.punctuation))
    text = re.sub(r"\b(a|an|the)\b", " ", text)
    return " ".join(text.split())


def token_f1(prediction, truth):
    pred_tokens, true_tokens = normalize_answer(prediction).split(), normalize_answer(truth).split()
    common = collections.Counter(pred_tokens) & collections.Counter(true_tokens)
    n_same = sum(common.values())
    if n_same == 0:
        return 0.0
    precision, recall = n_same / len(pred_tokens), n_same / len(true_tokens)
    return 2 * precision * recall / (precision + recall)


def squad_metrics(predictions, examples):
    exact, f1 = [], []
    for example in examples:
        prediction, truths = predictions[example["id"]], example["answers"]["text"]
        exact.append(max(float(normalize_answer(prediction) == normalize_answer(t)) for t in truths))
        f1.append(max(token_f1(prediction, t) for t in truths))
    return {"exact_match": 100 * np.mean(exact), "f1": 100 * np.mean(f1)}

In [18]:
start = time.time()
m2_output = trainer.predict(eval_features.select_columns(MODEL_INPUTS))
m2_start_logits, m2_end_logits = m2_output.predictions
print(f"Model 2 inference on {len(eval_features):,} windows: {time.time() - start:.1f}s")

m1_predictions = extract_answers(eval_examples, eval_features, m1_start_logits, m1_end_logits)
m2_predictions = extract_answers(eval_examples, eval_features, m2_start_logits, m2_end_logits)

m1_scores = squad_metrics(m1_predictions, eval_examples)
m2_scores = squad_metrics(m2_predictions, eval_examples)

results = pd.DataFrame({
    "Model 1: frozen BERT + LogReg": m1_scores,
    "Model 2: full fine-tuning": m2_scores,
}).T.round(2)
results

Model 2 inference on 10,753 windows: 47.1s


,exact_match,f1
Model 1: frozen BERT + LogReg,14.35,24.57
Model 2: full fine-tuning,69.25,79.69


In [19]:
for example in eval_examples.select(range(5)):
    print(f"Q: {example['question']}")
    print(f"   gold   : {example['answers']['text'][0]}")
    print(f"   model 1: {m1_predictions[example['id']]}")
    print(f"   model 2: {m2_predictions[example['id']]}\n")

Q: Which NFL team represented the AFC at Super Bowl 50?
   gold   : Denver Broncos
   model 1: Super Bowl 50
   model 2: Denver Broncos

Q: Which NFL team represented the NFC at Super Bowl 50?
   gold   : Carolina Panthers
   model 1: National Football Conference (NFC) champion Carolina Panthers
   model 2: Denver Broncos

Q: Where did Super Bowl 50 take place?
   gold   : Santa Clara, California
   model 1: 24
   model 2: Santa Clara, California

Q: Which NFL team won Super Bowl 50?
   gold   : Denver Broncos
   model 1: Super Bowl 50
   model 2: Denver Broncos

Q: What color was used to emphasize the 50th anniversary of the Super Bowl?
   gold   : gold
   model 1: golden anniversary"
   model 2: gold



## 7. Hub Publishing

In [20]:
import joblib
from huggingface_hub import EvalResult, HfApi, ModelCard, ModelCardData

# Model 1 (baseline) classifiers, saved locally to be uploaded with Model 2
START_CLASSIFIER_FILE = "model1_start_classifier.joblib"
END_CLASSIFIER_FILE = "model1_end_classifier.joblib"
joblib.dump(start_classifier, START_CLASSIFIER_FILE)
joblib.dump(end_classifier, END_CLASSIFIER_FILE)
print(f"Model 1 classifiers saved to '{START_CLASSIFIER_FILE}' and '{END_CLASSIFIER_FILE}'")

Model 1 classifiers saved to 'model1_start_classifier.joblib' and 'model1_end_classifier.joblib'


In [21]:
REPO_NAME = "bert-base-uncased-squad-qa"
COLLABORATOR = "Dexterg83"

api = HfApi()
repo_id = f"{api.whoami()['name']}/{REPO_NAME}"   # requires `hf auth login`
print("target repository:", repo_id)

target repository: Giobbva/bert-base-uncased-squad-qa


In [22]:
card_data = ModelCardData(
    language="en",
    license="apache-2.0",
    library_name="transformers",
    pipeline_tag="question-answering",
    base_model=MODEL_NAME,
    datasets=["rajpurkar/squad"],
    tags=["question-answering", "bert", "squad"],
    model_name=REPO_NAME,
    eval_results=[
        EvalResult(task_type="question-answering", dataset_type="rajpurkar/squad",
                   dataset_name="SQuAD v1.1", dataset_split="validation",
                   metric_type=name, metric_value=round(float(value), 2))
        for name, value in m2_scores.items()
    ],
)

card_text = f'''---
{card_data.to_yaml()}
---

# {REPO_NAME}

`{MODEL_NAME}` fine-tuned for extractive question answering on {N_TRAIN_EXAMPLES:,} examples of SQuAD v1.1.

| Model | Exact Match | F1 |
|---|---|---|
| Frozen BERT + Logistic Regression | {m1_scores["exact_match"]:.2f} | {m1_scores["f1"]:.2f} |
| **This model (full fine-tuning)** | **{m2_scores["exact_match"]:.2f}** | **{m2_scores["f1"]:.2f}** |

Evaluated on the full SQuAD v1.1 validation set ({len(eval_examples):,} questions).

## Files in this repository
- **Model 2 (full fine-tuning):** `BertForQuestionAnswering` weights, config and tokenizer.
- **Model 1 (feature-based baseline):** two scikit-learn Logistic Regression models
  (`StandardScaler` + `LogisticRegression` pipelines), hosted as `.joblib` files:
  - `{START_CLASSIFIER_FILE}`: scores each context word as the answer start
  - `{END_CLASSIFIER_FILE}`: scores each context word as the answer end

  They take as input the `last_hidden_state` (768-d) of the **frozen, original** `{MODEL_NAME}`,
  not of the fine-tuned model, at the first sub-token of each context word.

## Training
- AdamW, encoder lr {ENCODER_LR}, QA head lr {HEAD_LR}, weight decay 0.01, linear schedule with 10% warm-up
- {training_args.num_train_epochs:g} epochs, batch size {training_args.per_device_train_batch_size}, max length {MAX_LENGTH}, stride {STRIDE}, `fp16=True`, seed {SEED}

## Usage
```python
from transformers import pipeline
qa = pipeline("question-answering", model="{repo_id}")
qa(question="Where is the Eiffel Tower?", context="The Eiffel Tower is in Paris.")
```

Loading the baseline classifiers:
```python
import joblib
from huggingface_hub import hf_hub_download
start_classifier = joblib.load(hf_hub_download("{repo_id}", "{START_CLASSIFIER_FILE}"))
end_classifier = joblib.load(hf_hub_download("{repo_id}", "{END_CLASSIFIER_FILE}"))
```
'''

model_card = ModelCard(card_text)
model_card.save("MODEL_CARD.md")
print(card_text)

---
base_model: bert-base-uncased
datasets:
- rajpurkar/squad
language: en
library_name: transformers
license: apache-2.0
pipeline_tag: question-answering
tags:
- question-answering
- bert
- squad
model-index:
- name: bert-base-uncased-squad-qa
  results:
  - task:
      type: question-answering
    dataset:
      name: SQuAD v1.1
      type: rajpurkar/squad
      split: validation
    metrics:
    - type: exact_match
      value: 69.25
    - type: f1
      value: 79.69
---

# bert-base-uncased-squad-qa

`bert-base-uncased` fine-tuned for extractive question answering on 15,000 examples of SQuAD v1.1.

| Model | Exact Match | F1 |
|---|---|---|
| Frozen BERT + Logistic Regression | 14.35 | 24.57 |
| **This model (full fine-tuning)** | **69.25** | **79.69** |

Evaluated on the full SQuAD v1.1 validation set (10,570 questions).

## Files in this repository
- **Model 2 (full fine-tuning):** `BertForQuestionAnswering` weights, config and tokenizer.
- **Model 1 (feature-based baseline):** t

In [23]:
api.create_repo(repo_id, repo_type="model", exist_ok=True)
api.update_repo_settings(repo_id, gated="manual")   # restrict downloads before uploading

best_model = BertForQuestionAnswering.from_pretrained(FINAL_MODEL_DIR)
best_model.push_to_hub(repo_id, commit_message="Add fine-tuned bert-base-uncased QA model")
tokenizer.push_to_hub(repo_id, commit_message="Add tokenizer")
for classifier_file in [START_CLASSIFIER_FILE, END_CLASSIFIER_FILE]:
    api.upload_file(
        path_or_fileobj=classifier_file,
        path_in_repo=classifier_file,
        repo_id=repo_id,
        commit_message=f"Add Model 1 baseline classifier ({classifier_file})",
    )
model_card.push_to_hub(repo_id, commit_message="Add model card")

try:
    api.grant_access(repo_id, COLLABORATOR)
    print(f"access granted to '{COLLABORATOR}'")
except Exception as err:
    print(f"could not grant access automatically ({err}); "
          f"do it at https://huggingface.co/{repo_id}/settings")

print(f"published: https://huggingface.co/{repo_id}")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  7.39it/s]
Processing Files (1 / 1): 100%|██████████|  436MB /  436MB, 1.20MB/s  






Processing Files (1 / 1): 100%|██████████|  436MB /  436MB, 1.16MB/s  
New Data Upload: 100%|██████████|  436MB /  436MB, 1.16MB/s  
Validating: ██████████| 100%            
Processing Files (1 / 1): 100%|██████████| 22.9kB / 22.9kB, 2.27kB/s  






Processing Files (1 / 1): 100%|██████████| 22.9kB / 22.9kB, 2.12kB/s  
New Data Upload: 100%|██████████| 22.9kB / 22.9kB, 2.12kB/s  
Validating: ██████████| 100%            
Processing Files (1 / 1): 100%|██████████| 22.9kB / 22.9kB, 2.27kB/s  


Processing Files (1 / 1): 100%|██████████| 22.9kB / 22.9kB, 2.20kB/s  
New Data Upload: 100%|██████████| 22.9kB / 22.9kB, 2.20kB/s  
Validating: ██████████| 100%            


could not grant access automatically (401 Client Error. (Request ID: Root=1-6ab75cbe-197294fb1808a6fb410913fa;47dda9a0-6268-4820-abd6-cd3f1e2dfaa8)

Repository Not Found for url: https://huggingface.co/api/models/Giobbva/bert-base-uncased-squad-qa/user-access-request/grant.
Please make sure you specified the correct `repo_id` and `repo_type`.
If you are trying to access a private or gated repo, make sure you are authenticated and your token has the required permissions.
For more details, see https://huggingface.co/docs/huggingface_hub/authentication
Invalid username or password.); do it at https://huggingface.co/Giobbva/bert-base-uncased-squad-qa/settings
published: https://huggingface.co/Giobbva/bert-base-uncased-squad-qa
